# Brightest-string image: event display

Shows the per-event image the classifier's CNN branch receives (`trainer_domenc.brightest_string_image`, called directly here,
so this is exactly the network input):

* **channel 0** – brightest DOM ± 3 DOMs on its string;
* **channel 1** – the neighbouring string with the most charge at the brightest DOM's depth (±56 m), centred on its DOM nearest
  that depth — events between strings share their light over two strings;
* both in 5 ns bins relative to the brightest DOM's first pulse, charge / brightest-DOM charge, scaled as `log10(1 + 1e3 q) / 3`;
* 4 scalars: string-2 distance (/125 m), string-2 / string-1 charge ratio at that depth, string-2 depth offset (/17 m), has-string-2.

Events go through the training preprocessing (`trainer.preprocess_event_hits`, 0.1 PE threshold, 1500 ns window, validity cut).

Sections: 1 setup → 2 load → 3 single-event display → 4 single- vs two-string examples → 5 mean images & difference → 6 scalars.

In [ ]:
import os
N_THREADS = 8   # dev nodes kill processes over their CPU-time limit
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, str(N_THREADS))

import sqlite3
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

ANA = "/mnt/scratch/baburish/doublepulse/gnn/Analysis"
sys.path.insert(0, ANA)
import trainer_domenc as td
from trainer import build_geo_dict_kdtree, preprocess_event_hits

DBS = {  # random 40k-per-flavour samples used for training
    "nutau": f"{ANA}/data/combined_nutau_5TeV_40k.db",
    "nue":   f"{ANA}/data/combined_nue_5TeV_40k.db",
}
N_PER_FLAVOR = 2000      # random events per flavour to load
SEED = 3
CHARGE_THRESHOLD = 0.1   # train.py default

In [ ]:
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
COL = {"nutau": "#2a78d6", "nue": "#eb6834"}
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
SEQ.set_bad(SURFACE)
DIV = LinearSegmentedColormap.from_list(
    "div", ["#a8302f", "#e34948", "#f2a3a2", "#f0efec", "#9ec5f4", "#2a78d6", "#184f95"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

OFFSETS = np.arange(-td.IMG_N_SIDE, td.IMG_N_SIDE + 1)
EDGES_T = td.IMG_T_LO + td.IMG_DT * np.arange(td.IMG_N_BINS + 1)
CENTERS_T = 0.5 * (EDGES_T[1:] + EDGES_T[:-1])
EXTENT = [td.IMG_T_LO, td.IMG_T_HI, td.IMG_N_SIDE + 0.5, -td.IMG_N_SIDE - 0.5]   # top row = above
CH_NAME = ["ch 0: brightest string", "ch 1: neighbouring string"]


def to_charge(img):
    """Undo the input scaling: charge / brightest-DOM charge per 5 ns bin."""
    return (10 ** (3 * np.asarray(img, float)) - 1) / 1e3

## 1–2. Geometry and random events (training preprocessing)

In [ ]:
geo = pd.read_csv(f"{ANA}/geometry_clean.csv")
kd_tree, sid_geo = build_geo_dict_kdtree(geo)
td.set_geometry(kd_tree, sid_geo)
STRING_XY = geo.groupby("string")[["dom_x", "dom_y"]].first()


class SerialKD:
    """trainer.map_pulses_to_strings queries with workers=-1 (all cores, per event); same query, one thread."""
    def __init__(self, tree):
        self.tree = tree

    def query(self, pts, workers=1, **kw):
        return self.tree.query(pts, workers=1, **kw)


def load_flavor(flavor, n, seed=SEED):
    con = sqlite3.connect(f"file:{DBS[flavor]}?mode=ro", uri=True)
    info = pd.read_sql_query(
        "SELECT t.event_no, t.zenith, t.primary_energy, r.cascade_reco_energy_tev "
        "FROM truth t JOIN reco r USING (event_no)", con).set_index("event_no")
    pick = np.sort(np.random.default_rng(seed).choice(info.index.to_numpy(), min(n, len(info)), replace=False))
    con.execute("CREATE TEMP TABLE pick(event_no INTEGER PRIMARY KEY)")
    con.executemany("INSERT INTO pick VALUES (?)", [(int(e),) for e in pick])
    p = pd.read_sql_query(
        "SELECT event_no, dom_x, dom_y, dom_z, dom_time, charge FROM CleanedROIPulses "
        "WHERE event_no IN (SELECT event_no FROM temp.pick) ORDER BY event_no", con)
    con.close()

    arr = p.to_numpy(float)
    ev_ids, starts = np.unique(arr[:, 0], return_index=True)
    bounds = np.r_[starts, len(arr)]
    rows = []
    for i, ev_no in enumerate(ev_ids):
        x, y, z, t, q, sid = preprocess_event_hits(arr[bounds[i]:bounds[i + 1]], SerialKD(kd_tree), sid_geo,
                                                   charge_threshold=CHARGE_THRESHOLD)
        if len(t) < 4 or len(np.unique(sid)) < 2:     # training validity cut
            continue
        img, sc, img_info = td.brightest_string_image(x, y, z, t, q, return_info=True)
        # charge per string (for the top-view panel)
        q_str = pd.Series(q).groupby(sid).sum()
        inf = info.loc[int(ev_no)]
        rows.append(dict(event_no=int(ev_no), flavor=flavor, img=img, scalars=sc, q_string=q_str, info=img_info,
                         q_total=q.sum(), coszen=np.cos(inf["zenith"]), log_e=np.log10(inf["primary_energy"]),
                         reco_e_tev=inf["cascade_reco_energy_tev"]))
    print(f"{flavor}: {len(rows)} events")
    return rows


EVENTS = {fl: load_flavor(fl, N_PER_FLAVOR) for fl in DBS}
ALL = pd.DataFrame([{k: v for k, v in e.items() if k not in ("img", "q_string", "info", "scalars")}
                    | dict(zip(td.IMG_SCALAR_NAMES, e["scalars"]))
                    for evs in EVENTS.values() for e in evs])
print(f"image shape {EVENTS['nutau'][0]['img'].shape}, scalars {td.IMG_SCALAR_NAMES}")

## 3. Single-event display
`show_event(flavor, i)` — top: the two image channels exactly as fed to the CNN; middle: per-DOM charge vs time for each channel
(rows stacked, top = above, each scaled to the brightest DOM's peak bin); bottom-left: top view of the strings (marker area ∝ string charge,
blue = brightest string, orange = neighbouring string used for channel 1).

In [ ]:
def show_event(flavor, i):
    ev = EVENTS[flavor][i]
    img, sc = ev["img"], ev["scalars"]
    raw = to_charge(img)
    fig = plt.figure(figsize=(16, 9.5))
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1.25, 1.1], width_ratios=[1, 1, 0.05])

    for ch in range(2):
        ax = fig.add_subplot(gs[0, ch])
        im = ax.imshow(np.ma.masked_less_equal(img[ch], 0), aspect="auto", extent=EXTENT, cmap=SEQ,
                       vmin=0, vmax=max(img.max(), 1e-6), interpolation="nearest")
        ax.set_yticks(OFFSETS); ax.grid(False)
        ax.set_title(f"{CH_NAME[ch]} (network input)", loc="left")
        ax.set_xlabel("time since brightest DOM's first pulse [ns]")
        if ch == 0:
            ax.set_ylabel("DOM offset (− above)")
    fig.colorbar(im, cax=fig.add_subplot(gs[0, 2]), label="log10(1 + 1e3 q/Q_b) / 3")

    peak = raw[0, td.IMG_N_SIDE].max() or 1.0
    for ch in range(2):
        ax = fig.add_subplot(gs[1, ch])
        for k, o in enumerate(OFFSETS):
            base = -k * 1.1
            ax.step(CENTERS_T, raw[ch, k] / peak + base, where="mid", linewidth=1.4,
                    color=INK if (ch == 0 and o == 0) else (COL["nutau"] if ch == 0 else COL["nue"]))
            ax.text(td.IMG_T_HI + 8, base + 0.05, f"{o:+d}" if (o or ch) else "brightest", fontsize=8, color=INK_2)
        ax.set_yticks([]); ax.set_xlim(td.IMG_T_LO, td.IMG_T_HI + 90)
        ax.set_title(f"{CH_NAME[ch]}: q vs t per DOM (÷ brightest DOM's peak bin)", loc="left")
        ax.set_xlabel("time since brightest DOM's first pulse [ns]")

    ax = fig.add_subplot(gs[2, 0])
    ax.scatter(STRING_XY["dom_x"], STRING_XY["dom_y"], s=6, color=GRID, zorder=1)
    q_str = ev["q_string"]
    xy = STRING_XY.loc[q_str.index]
    ax.scatter(xy["dom_x"], xy["dom_y"], s=600 * q_str / q_str.max(), color=INK_2, alpha=0.3, zorder=2, linewidths=0)
    info = ev["info"]
    for key, color, name in [("string1", COL["nutau"], "ch 0 string"), ("string2", COL["nue"], "ch 1 string")]:
        if info[key] is not None:
            sx, sy = STRING_XY.loc[info[key]]
            ax.scatter([sx], [sy], s=140, facecolors="none", edgecolors=color, linewidths=2.5, zorder=3,
                       label=f"{name} ({info[key]})")
    ax.legend(loc="upper left", fontsize=8)
    ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title("top view: charge per string (area)", loc="left")

    ax = fig.add_subplot(gs[2, 1]); ax.axis("off")
    txt = (f"{flavor}  event {ev['event_no']}\n"
           f"true cos zenith {ev['coszen']:+.2f}   true log10 E {ev['log_e']:.2f}   reco E {ev['reco_e_tev']:.1f} TeV\n"
           f"total charge {ev['q_total']:.0f} PE\n\n"
           f"string 2: distance {sc[0] * 125:.0f} m   charge ratio {sc[1]:.2f}   "
           f"depth offset {sc[2] * 17:+.0f} m   present {bool(sc[3])}")
    ax.text(0, 0.9, txt, va="top", fontsize=11, family="monospace")
    fig.tight_layout()
    plt.show()


show_event("nutau", 0)
show_event("nue", 0)

## 4. Single-string vs two-string topologies
Events whose light is mostly on one string (string-2 charge ratio < 0.15) vs shared between two neighbouring strings (ratio > 0.6).

In [ ]:
def pick(flavor, cond, n=2, seed=SEED):
    idx = [i for i, e in enumerate(EVENTS[flavor]) if cond(e["scalars"])]
    return list(np.random.default_rng(seed).choice(idx, min(n, len(idx)), replace=False))


single = lambda s: s[3] == 1 and s[1] < 0.15
double = lambda s: s[3] == 1 and s[1] > 0.6
for fl in EVENTS:
    frac_single = np.mean([single(e["scalars"]) for e in EVENTS[fl]])
    frac_double = np.mean([double(e["scalars"]) for e in EVENTS[fl]])
    print(f"{fl}: single-string-like {frac_single:.0%}, two-string-like {frac_double:.0%}")

fig, axes = plt.subplots(4, 2, figsize=(15, 11), sharex=True)
r = 0
for fl in EVENTS:
    for label, cond in [("single-string", single), ("two-string", double)]:
        i = pick(fl, cond, 1)[0]
        ev = EVENTS[fl][i]
        for ch in range(2):
            ax = axes[r, ch]
            ax.imshow(np.ma.masked_less_equal(ev["img"][ch], 0), aspect="auto", extent=EXTENT, cmap=SEQ,
                      vmin=0, vmax=ev["img"].max(), interpolation="nearest")
            ax.set_yticks(OFFSETS); ax.grid(False)
            ax.set_title(f"{fl} {label} (event {ev['event_no']}, ratio {ev['scalars'][1]:.2f}) — {CH_NAME[ch]}",
                         loc="left", fontsize=9)
        r += 1
for ax in axes[-1]:
    ax.set_xlabel("time since brightest DOM's first pulse [ns]")
for ax in axes[:, 0]:
    ax.set_ylabel("DOM offset")
fig.tight_layout(); plt.show()

## 5. Mean images per flavour and nutau − nue difference
Mean of the network input per pixel (both channels), and the per-pixel significance
`z = (mean_tau − mean_e) / sqrt(var_tau/n_tau + var_e/n_e)`. Note: these samples are **not** direction-matched, so part of any
difference reflects the different direction distributions of the nutau and nue simulation sets.

In [ ]:
stack = {fl: np.stack([e["img"] for e in evs]) for fl, evs in EVENTS.items()}
m1, m0 = stack["nutau"].mean(0), stack["nue"].mean(0)
z = (m1 - m0) / np.sqrt(stack["nutau"].var(0) / len(stack["nutau"]) + stack["nue"].var(0) / len(stack["nue"]) + 1e-30)

fig, axes = plt.subplots(2, 3, figsize=(17, 7), sharex=True, sharey=True)
vmax = max(m1.max(), m0.max())
for ch in range(2):
    for c, (fl, m) in enumerate([("nutau", m1), ("nue", m0)]):
        im = axes[ch, c].imshow(m[ch], aspect="auto", extent=EXTENT, cmap=SEQ, vmin=0, vmax=vmax, interpolation="nearest")
        axes[ch, c].set_title(f"mean input: {fl}, {CH_NAME[ch]}", loc="left", fontsize=10)
    zim = axes[ch, 2].imshow(np.clip(z[ch], -6, 6), aspect="auto", extent=EXTENT, cmap=DIV,
                             norm=TwoSlopeNorm(0, -6, 6), interpolation="nearest")
    axes[ch, 2].set_title(f"z, ch {ch} (blue = nutau excess)", loc="left", fontsize=10)
    axes[ch, 0].set_ylabel("DOM offset (− above)")
for ax in axes.ravel():
    ax.grid(False); ax.set_yticks(OFFSETS)
for ax in axes[-1]:
    ax.set_xlabel("time since brightest DOM's first pulse [ns]")
fig.colorbar(im, ax=axes[:, :2], label="mean input value", shrink=0.8)
fig.colorbar(zim, ax=axes[:, 2], label="z (clipped ±6)", shrink=0.8)
plt.show()
for ch in range(2):
    print(f"{CH_NAME[ch]}: pixels with |z| > 3: {(np.abs(z[ch]) > 3).sum()} of {z[ch].size} "
          f"(≈{0.0027 * z[ch].size:.1f} expected from noise)")

## 6. String-2 scalars per flavour

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
specs = [("string2_dist_125m", 125, "string-2 distance [m]", np.linspace(0, 300, 31)),
         ("string2_q_ratio", 1, "string-2 / string-1 charge at depth", np.linspace(0, 1.5, 31)),
         ("string2_dz_17m", 17, "string-2 centre depth offset [m]", np.linspace(-60, 60, 25))]
for ax, (col, scale, xlabel, bins) in zip(axes, specs):
    for fl in EVENTS:
        sub = ALL[(ALL["flavor"] == fl) & (ALL["has_string2"] == 1)]
        ax.hist(sub[col] * scale, bins=bins, histtype="step", density=True, linewidth=1.8, color=COL[fl], label=fl)
    ax.set_xlabel(xlabel)
axes[0].set_ylabel("density"); axes[0].legend(loc="upper right")
fig.tight_layout(); plt.show()
display(ALL.groupby("flavor")[td.IMG_SCALAR_NAMES].describe().T.round(3))